# ChemBreak 12 — Fixed Partition and Run Guards

This notebook establishes the **data contract that must exist before training**.

ChemBreak 12 uses the frozen 500-task bank and a permanent six-way partition:

- **Train: 241**
- **Test1: 50**
- **Test2: 50**
- **Test3: 50**
- **Test4: 50**
- **Reserve: 59**

The 59 Reserve rows are not sampled by CB12; they are the rows already marked `is_reserve=True` in the source bank.

The central rule is simple: **training code may see Train only**. Test1-Test4 remain held out, and Reserve is a contingency pool rather than an ordinary test set.


## Cell 1 — Locate the CB12 repository

Run the notebook from the repository root. In Colab Enterprise, clone/upload the repository first and then set `PROJECT_ROOT` to that folder. CB12 does not point to ChemBreak 7–11 storage or checkpoints.


In [ ]:
from pathlib import Path
import os, sys

# Preferred: run this notebook from the chembreak12 repository root.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "chembreak12").exists():
    # If the notebook is opened from notebooks/, move one directory up.
    candidate = PROJECT_ROOT.parent
    if (candidate / "src" / "chembreak12").exists():
        PROJECT_ROOT = candidate

assert (PROJECT_ROOT / "src" / "chembreak12").exists(), (
    "Set PROJECT_ROOT to the chembreak12 repository folder before continuing."
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)


## Cell 2 — Verify the frozen source bank

This loads the uploaded/frozen source bank and enforces the basic invariants: exactly 500 rows, unique `assignment_id`, unique normalized benchmark prompts, and valid Reserve flags. It also computes a SHA-256 for every benchmark prompt so later prompt edits are detectable.


In [ ]:
from chembreak12.dataset import load_task_bank

SOURCE = PROJECT_ROOT / "data" / "final_task_bank.csv"
source = load_task_bank(SOURCE)
print("rows:", len(source))
print("primary:", int((~source["is_reserve"]).sum()))
print("reserve:", int(source["is_reserve"].sum()))
print("unique assignment IDs:", source["assignment_id"].nunique())
print("unique prompt hashes:", source["prompt_sha256"].nunique())


## Cell 3 — Partition creation (construction-time only)

The released repository already contains the locked manifest. This cell demonstrates how the manifest was constructed.

**Important:** once CB12 training begins, do not use this cell to create a different split. The existing manifest becomes part of the experiment record. If the partition design is deliberately changed, create a new protocol/version instead.


In [ ]:
from chembreak12.partition import build_manifest

# Determinism demonstration only: regenerate in memory and compare with the locked file.
import pandas as pd
locked_manifest = pd.read_csv(PROJECT_ROOT / "data" / "CB12_partition_manifest_v1.csv")
regenerated = build_manifest(source)

same = regenerated.equals(locked_manifest)
print("Regenerated manifest exactly matches locked manifest:", same)
assert same


## Cell 4 — Verify the cryptographic partition lock

This is the cell that should run **before every training or evaluation job**. It checks the source file hash, canonical dataset hash, manifest hash, protocol ID, fixed seed, exact split counts, prompt hashes, Reserve preservation, complete coverage, and pairwise disjointness.


In [ ]:
from chembreak12.integrity import verify_lock

MANIFEST = PROJECT_ROOT / "data" / "CB12_partition_manifest_v1.csv"
LOCK = PROJECT_ROOT / "data" / "CB12_partition_lock_v1.json"
manifest = pd.read_csv(MANIFEST)

result = verify_lock(
    source_path=SOURCE,
    frame=source,
    manifest_path=MANIFEST,
    manifest=manifest,
    lock_path=LOCK,
)
print(result)


## Cell 5 — Confirm the six fixed partitions

These counts are part of the CB12 protocol. A run should stop if they differ.


In [ ]:
expected = {"Train": 241, "Test1": 50, "Test2": 50, "Test3": 50, "Test4": 50, "Reserve": 59}
observed = manifest["split"].value_counts().to_dict()
print(observed)
assert observed == expected


## Cell 6 — Inspect distribution balance without opening task text

The partition algorithm balances `hc_id`, `hd_id`, and `ot_id`. This cell displays category counts only, so you can check the experimental distribution without using held-out prompt text during training.


In [ ]:
joined = source[["assignment_id", "hc_id", "hd_id", "ot_id"]].merge(
    manifest[["assignment_id", "split"]], on="assignment_id", validate="one_to_one"
)
for column in ["hc_id", "hd_id", "ot_id"]:
    print("\n===", column, "===")
    print(pd.crosstab(joined[column], joined["split"]).to_string())


## Cell 7 — Choose the run mode and split

This is the experiment firewall.

- `MODE="train"` permits only `SPLIT="Train"`.
- `MODE="eval"` permits only Test1-Test4.
- `MODE="reserve"` permits only Reserve and requires a documented contingency reason.

If a configuration accidentally asks training code for Test1, the guard raises an error before the split is loaded.


In [ ]:
from chembreak12.guards import RunAccess, validate_run_access

MODE = "train"       # train | eval | reserve
SPLIT = "Train"      # Train | Test1 | Test2 | Test3 | Test4 | Reserve
RESERVE_REASON = None # required only for MODE="reserve"

access = RunAccess(mode=MODE, split=SPLIT, reserve_reason=RESERVE_REASON)
validate_run_access(access)
print("ACCESS ALLOWED:", access)


## Cell 8 — Load exactly one authorized split

Only after the lock and access guard pass do we load the selected CSV. This prevents a training job from casually starting with the entire 500-task bank.


In [ ]:
split_path = PROJECT_ROOT / "data" / "splits" / f"CB12_{SPLIT}.csv"
run_tasks = pd.read_csv(split_path)
print("loaded split:", SPLIT)
print("task count:", len(run_tasks))
print("first task IDs:", run_tasks["assignment_id"].head().tolist())


## Cell 9 — Create an experiment revision record

Record the partition hash and run-access choice with every experiment revision. This makes it possible to prove later that two runs used the same fixed partition.


In [ ]:
import json, hashlib
from datetime import datetime, timezone

EXPERIMENT_REVISION = "CB12-R001"
record = {
    "experiment_revision": EXPERIMENT_REVISION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "mode": MODE,
    "split": SPLIT,
    "task_count": len(run_tasks),
    "manifest_sha256": hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
    "partition_protocol": "CB12_PARTITION_V1",
    "partition_seed": 12026,
}

out_dir = PROJECT_ROOT / "outputs" / EXPERIMENT_REVISION
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "run_access.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
print(json.dumps(record, indent=2))


## Cell 10 — Model/policy integration boundary

At this point the **dataset side is frozen and safe from accidental leakage**. An approved evaluation runner should accept `run_tasks` rather than opening the full source bank itself.

For training, the runner receives the 241 Train rows only. After policy/model-selection choices are frozen, held-out evaluation can be run using a separate notebook/job with `MODE="eval"` and one of Test1-Test4.

CB12's partition package intentionally keeps adversarial prompt-generation logic outside this data-governance layer. This prevents partition code from silently becoming an optimization path into held-out harmful task content.


## What to archive with a CB12 paper/report

Archive the source-bank checksum, partition manifest, partition lock, code commit, model revisions, random seeds, training configuration, frozen policy/configuration, evaluation configuration, and result files. That gives an external reviewer enough information to reproduce the data split and verify that it was fixed before training.
